In [1]:
from embedder import Embedder

2026-06-22 10:53:46.633483649 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


In [2]:
embedder= Embedder()

In [3]:
vector = embedder.encode("How does approximate nearest neighbor search work?")

In [4]:
vector.shape

(384,)

In [5]:
vector[0]

np.float64(-0.02058203437252893)

In [6]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [7]:
for i, doc in enumerate(documents):
    if "07-sqlitesearch-vector" in doc["filename"]:
        target_doc = doc
        target_index = i
        break

In [8]:
texts = [doc["filename"] + " " + doc["content"] for doc in documents]

In [9]:
texts[target_index]

'02-vector-search/lessons/07-sqlitesearch-vector.md # Vector Search with sqlitesearch\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=csxKescwJYM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous section we used minsearch for vector search.\n\nIt works, but it has three problems:\n\n- It rebuilds the index on every startup\n- It keeps everything in memory\n- It searches by brute force\n\n\nWith text search we never felt these. Indexing was fast because we\ndidn\'t embed anything. With vector search, indexing runs a neural\nnetwork over every document, so it takes a minute on our dataset.\nKeeping everything in memory is fine here, but a larger dataset would\nneed too much space.\n\nThe third problem is brute-force search. For every query we compare the\nquery vector against every single document. With 1,000 documents this is\nfine, probably even faster than anything smarter. But as the dataset\ngrows past 10,000 or so, it gets slow, and we\'ll want an approximat

In [10]:
from sklearn.metrics.pairwise import cosine_similarity

In [11]:
import numpy as np

similarity = np.dot(embedder.encode(texts[target_index]), vector)
similarity

np.float64(0.37509004576991944)

In [12]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [13]:
texts_chunks = [chunk["filename"] + " " + chunk["content"] for chunk in chunks]

In [14]:
encoded_chunks = embedder.encode_batch(texts_chunks)

In [15]:
scores = encoded_chunks.dot(vector)

In [48]:
scores[:5]

array([ 0.32162674,  0.16601091,  0.0584318 , -0.04033262,  0.05768674])

In [17]:
top_indices = np.argsort(scores)[::-1][:5]  # Top 5
for idx in top_indices:
    print(f"Index: {idx}, Score: {scores[idx]}")
    print(f"Chunk: {texts_chunks[idx]}\n")

Index: 94, Score: 0.5685932142582364
Chunk: 02-vector-search/lessons/07-sqlitesearch-vector.md rch. We score
the query against every document and pick the top ones. It always finds
the true top matches, but it pays for that by touching everything.

Approximate nearest neighbor (ANN) search takes a shortcut. Instead of
comparing against everything, it first narrows down to a region of
likely matches. Then it scores only within that region. It may miss the
absolute best match, but the results are still good and it's much
faster.

```text
NN (exact):    compare query against ALL documents -> top 5
ANN (approx):  narrow down to a region -> compare within region -> top 5
```

## sqlitesearch

sqlitesearch is the persistent sibling of minsearch, and it solves both
problems at once.

We already used it in module 1 for persistent text search. It also does
vector search through its `VectorSearchIndex` class. It stores vectors
in SQLite, a real on-disk database, and uses ANN strategies for
retri

In [18]:
from minsearch import VectorSearch

In [19]:
vindex = VectorSearch(keyword_fields=["filename"])

In [20]:
query = "What metric do we use to evaluate a search engine?"

In [21]:
vindex.fit(encoded_chunks, chunks)  # Pass vectors + chunks, not texts + documents

query_vector = embedder.encode(query)
results = vindex.search(query_vector, num_results=5)

In [22]:
results

[{'start': 0,
  'content': "# Search Evaluation Metrics\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=TuirMy3Pdbk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we computed relevance lists for search results.\nWe can turn those lists into metrics.\n\n## Hit Rate\n\nHit Rate (also called Recall@k) measures the fraction of queries where\nthe correct document appears anywhere in the results:\n\n```python\nexample = [\n    [1, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 1, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n]\n```\n\nEach line is one query. If a line contains `1`, search found the\ncorrect document somewhere in the top 5 results. If the line contains\nonly zeros, search did not find the correct document.\n\nIn our set

In [33]:
from minsearch import Index

In [34]:
index = Index(
    text_fields=["filename"],
    keyword_fields=["course"]
    )
index.fit(chunks)


In [35]:
query = "How do I store vectors in PostgreSQL?"

In [36]:
search_results = index.search(query,
             num_results=10
             )

In [37]:
search_results

[{'start': 0,
  'content': '# Built-in Judge\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=YLOLQyrMDuY&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous module we used an LLM as a judge for offline evaluation.\nOne LLM grades the output of another. We can run the same idea online.\n\nAfter each answer, we ask a judge whether it\'s relevant to the question.\nThat gives us an automatic quality signal on every response. We don\'t\nhave to wait for anyone to click thumbs up or down.\n\nThere\'s one real difference from the offline setup. Back then we had\nground truth: a reference answer to compare against. The judge could\ncheck our answer against that known-good one. Online we don\'t have it.\n\nThe judge now only sees the question and the answer. It has to make the\ncall on its own. That\'s a harder job, so in the instructions we describe\nmore carefully what a good answer looks like.\n\n## Adding relevance evaluation\n\nWe use structured output so the judge re

In [39]:
query_vector = embedder.encode(query)
results = vindex.search(query_vector, num_results=10)

In [40]:
results

[{'start': 0,
  'content': '# Vector Search with PGVector\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=0P54MFyz-mc&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nMany real databases can do vector search. Elasticsearch has it, and\nthere are dedicated stores like Qdrant and Chroma. We\'ll go with\nPostgres. Most of us already run it at work, and the data engineering\ncourse uses it too. The concept is the same as with sqlitesearch, only\nthe database under the hood changes.\n\n[pgvector](https://github.com/pgvector/pgvector) is the PostgreSQL\nextension that makes this work. Install it and Postgres can do vector\nsimilarity search. On top of that you get the usual production features,\nlike concurrent access, transactions, and large datasets.\n\nWe\'ll run Postgres with pgvector in Docker.\n\n## Starting Postgres with pgvector\n\nPull the image and start a container:\n\n```bash\ndocker run -it \\\n    --name pgvector \\\n    -e POSTGRES_USER=user \\\n    -e POSTGRES_PASSWO

In [41]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [42]:
last_query= "How do I give the model access to tools?"

In [43]:
query_vector = embedder.encode(last_query)

In [44]:
vector_results = vindex.search(query_vector, num_results=20)
text_results = index.search(query, num_results=20)

In [50]:
vector_results[0]

{'start': 2000,
 'content': ' num_results=5,\n        boost_dict={"question": 3.0, "section": 0.5},\n        filter_dict={"course": "llm-zoomcamp"}\n    )\n```\n\nThen register it without passing a schema:\n\n```python\nagent_tools = Tools()\nagent_tools.add_tool(search)\n```\n\nYou can look at what ToyAIKit produced.\n\n```python\nagent_tools.get_tools()\n```\n\nThe output is the same JSON schema we hand-wrote in the function\ncalling lesson. ToyAIKit generated it from the docstring and the type\nhint.\n\nEvery modern agent framework does this same trick. It reads a typed\nPython function with a docstring and builds the schema from it. The\nOpenAI Agents SDK, PydanticAI, LangChain and Google ADK all work this\nway. You write the tool and the framework figures out how to describe\nit.\n\n## The chat interface and runner\n\nCreate the chat interface and a callback, then build the runner:\n\n```python\nchat_interface = IPythonChatInterface()\ncallback = DisplayingRunnerCallback(chat_inte

In [51]:
text_results[0]

{'start': 0,
 'content': '# Built-in Judge\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=YLOLQyrMDuY&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous module we used an LLM as a judge for offline evaluation.\nOne LLM grades the output of another. We can run the same idea online.\n\nAfter each answer, we ask a judge whether it\'s relevant to the question.\nThat gives us an automatic quality signal on every response. We don\'t\nhave to wait for anyone to click thumbs up or down.\n\nThere\'s one real difference from the offline setup. Back then we had\nground truth: a reference answer to compare against. The judge could\ncheck our answer against that known-good one. Online we don\'t have it.\n\nThe judge now only sees the question and the answer. It has to make the\ncall on its own. That\'s a harder job, so in the instructions we describe\nmore carefully what a good answer looks like.\n\n## Adding relevance evaluation\n\nWe use structured output so the judge retu

In [45]:
results = rrf([vector_results, text_results])

In [46]:
results

[{'start': 2000,
  'content': ' num_results=5,\n        boost_dict={"question": 3.0, "section": 0.5},\n        filter_dict={"course": "llm-zoomcamp"}\n    )\n```\n\nThen register it without passing a schema:\n\n```python\nagent_tools = Tools()\nagent_tools.add_tool(search)\n```\n\nYou can look at what ToyAIKit produced.\n\n```python\nagent_tools.get_tools()\n```\n\nThe output is the same JSON schema we hand-wrote in the function\ncalling lesson. ToyAIKit generated it from the docstring and the type\nhint.\n\nEvery modern agent framework does this same trick. It reads a typed\nPython function with a docstring and builds the schema from it. The\nOpenAI Agents SDK, PydanticAI, LangChain and Google ADK all work this\nway. You write the tool and the framework figures out how to describe\nit.\n\n## The chat interface and runner\n\nCreate the chat interface and a callback, then build the runner:\n\n```python\nchat_interface = IPythonChatInterface()\ncallback = DisplayingRunnerCallback(chat_in